# 02 — Build an inspectable local RAG assistant

**Track:** Beginner · **Stage:** Local Implementation

By the end of this notebook, you will build, run, debug, and evaluate a dependency-free local RAG assistant using **LangChain**, **HuggingFace Embeddings**, and **Chroma**. You will explain every score it produces, build a bounded context window, and return citations.

> The goal is not to make a clever chatbot. It is to establish a transparent baseline where you can inspect the exact evidence retrieved before generating an answer.

## Setup: Local Models and Vector Database

We will use `HuggingFaceEmbeddings` for local, cost-free vectorization, and `Chroma` for our local vector store. For generation, we mock the LLM output to keep this notebook fully local and API-key free. This mirrors the major interfaces and execution stages of a production RAG pipeline while intentionally omitting production infrastructure and controls.

In [1]:
# !pip install langchain-core langchain-huggingface langchain-chroma sentence-transformers

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.language_models import FakeListLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## 1. Ingestion: Document to Vector

In a real system, documents are parsed, chunked, and enriched with metadata. Here, we define a small corpus manually to maintain full visibility. Notice we use rich metadata (like `document_id`, `chunk_id`, `title`) just as we did in Course 01.

In [2]:
corpus = [
    Document(
        page_content="Harborline Support Escalation Policy: Tier 1 may reboot edge nodes. Tier 2 must approve any database failover.",
        metadata={
            "document_id": "pol-esc-01", 
            "chunk_id": "pol-esc-01#perm-01",
            "title": "Support Escalation Policy",
            "source": "escalation_policy.md", 
            "section": "permissions",
            "version": "1.4"
        }
    ),
    Document(
        page_content="Incident Response: If the payments API returns 503, immediately check the Stripe gateway status page before paging on-call.",
        metadata={
            "document_id": "ir-pay-02", 
            "chunk_id": "ir-pay-02#api-01",
            "title": "Incident Response: Payments",
            "source": "incident_response.md", 
            "section": "payments",
            "version": "2.1"
        }
    ),
    Document(
        page_content="System Architecture: The payments API relies on a PostgreSQL cluster in the us-east-1 region.",
        metadata={
            "document_id": "arch-sys-03", 
            "chunk_id": "arch-sys-03#db-01",
            "title": "System Architecture",
            "source": "architecture.md", 
            "section": "database",
            "version": "1.0"
        }
    )
]

# Initialize local embeddings (downloads a small model the first time)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create an ephemeral local vector store
vectorstore = Chroma.from_documents(corpus, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

/Users/mahsateimourikia/.pyenv/versions/3.11.13/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10424.28it/s]

## 2. Inspecting Retrieval (The Evidence Boundary)

Before we pass anything to an LLM, we must verify the retriever finds the correct evidence. Dense embeddings capture semantic meaning, not just exact keyword matches.

In [3]:
question = "Who is allowed to trigger a DB failover?"

# Using similarity_search_with_score to inspect the raw distance metrics
results = vectorstore.similarity_search_with_score(question, k=2)

print(f"Question: {question}\n")
for rank, (doc, score) in enumerate(results, start=1):
    print(f"--- Rank {rank} ---")
    # Note: Chroma defaults to L2 distance (lower is closer/better). 
    # Do NOT interpret this score as answer confidence! It is highly backend/metric dependent.
    print(f"Raw Score/Distance: {score:.4f}")
    print(f"Title: {doc.metadata['title']}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Chunk ID: {doc.metadata['chunk_id']}")
    print(f"Content: {doc.page_content}\n")

Question: Who is allowed to trigger a DB failover?

--- Rank 1 ---
Raw Score/Distance: 1.1486
Title: Support Escalation Policy
Source: escalation_policy.md
Chunk ID: pol-esc-01#perm-01
Content: Harborline Support Escalation Policy: Tier 1 may reboot edge nodes. Tier 2 must approve any database failover.

--- Rank 2 ---
Raw Score/Distance: 1.5411
Title: System Architecture
Source: architecture.md
Chunk ID: arch-sys-03#db-01
Content: System Architecture: The payments API relies on a PostgreSQL cluster in the us-east-1 region.



## 3. Separating Candidates from Context

Retrieval produces **candidates**. Generation consumes **evidence context**. We should separate these stages.
First, we format the candidates into an evidence string using `<EVIDENCE>` tags with stable IDs.

In [4]:
evidence_mapping = {}

def format_docs_with_citations(docs):
    global evidence_mapping
    evidence_mapping.clear()
    
    formatted_chunks = []
    for i, d in enumerate(docs, start=1):
        evidence_id = f"E{i}"
        evidence_mapping[evidence_id] = d.metadata
        
        chunk = (
            f"<EVIDENCE id=\"{evidence_id}\">\n"
            f"Title: {d.metadata['title']}\n"
            f"Section: {d.metadata['section']}\n"
            f"Content:\n{d.page_content}\n"
            f"</EVIDENCE>"
        )
        formatted_chunks.append(chunk)
    return "\n\n".join(formatted_chunks)

candidates = retriever.invoke(question)
evidence_context = format_docs_with_citations(candidates)

print(evidence_context)

<EVIDENCE id="E1">
Title: Support Escalation Policy
Section: permissions
Content:
Harborline Support Escalation Policy: Tier 1 may reboot edge nodes. Tier 2 must approve any database failover.
</EVIDENCE>

<EVIDENCE id="E2">
Title: System Architecture
Section: database
Content:
System Architecture: The payments API relies on a PostgreSQL cluster in the us-east-1 region.
</EVIDENCE>


## 4. Constructing the Chain

Now we bind the evidence context to the generation step, instructing the LLM to cite its sources using the `[E#]` IDs.

In [5]:
prompt = ChatPromptTemplate.from_template(
    "Answer the user's question based strictly on the context below. "
    "If you cannot answer based on the evidence, say so. "
    "Cite the evidence IDs (e.g., [E1]) for your factual claims.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

llm = FakeListLLM(responses=["Based on the Support Escalation Policy, Tier 2 must approve any database failover [E1]."])

chain = prompt | llm | StrOutputParser()

print("Final Answer:")
print(chain.invoke({"context": evidence_context, "question": question}))

Final Answer:
Based on the Support Escalation Policy, Tier 2 must approve any database failover [E1].


## 5. Failure Cases

Retrieval returning *something* does not mean sufficient evidence exists. 

### Unsupported Question
Let's ask a question that our corpus has no answer for.

In [6]:
unsupported_question = "Who is Harborline's CEO?"
unsupported_candidates = retriever.invoke(unsupported_question)

print("Retrieved Candidates for Unsupported Question:")
for doc in unsupported_candidates:
    print(f"- {doc.metadata['title']}: {doc.page_content[:50]}...")


Retrieved Candidates for Unsupported Question:
- Support Escalation Policy: Harborline Support Escalation Policy: Tier 1 may r...
- System Architecture: System Architecture: The payments API relies on a ...


Even though the retriever brought back documents (because it just returns the closest vectors in embedding space), the generator should read this context and abstain. 

### Multiple-Source Question
Let's ask a question that requires assembling facts from multiple chunks.

In [7]:
multi_question = "If the payments API returns a 503, what should I check, and what region is the database in?"
multi_candidates = retriever.invoke(multi_question)
multi_context = format_docs_with_citations(multi_candidates)

print("Evidence for Multiple-Source Question:\n")
print(multi_context)

Evidence for Multiple-Source Question:

<EVIDENCE id="E1">
Title: Incident Response: Payments
Section: payments
Content:
Incident Response: If the payments API returns 503, immediately check the Stripe gateway status page before paging on-call.
</EVIDENCE>

<EVIDENCE id="E2">
Title: System Architecture
Section: database
Content:
System Architecture: The payments API relies on a PostgreSQL cluster in the us-east-1 region.
</EVIDENCE>


## 6. Debugging the Trace

If the answer is wrong, where did the pipeline fail? 
1. **Ingestion failure:** The document wasn't in the vector store.
2. **Retrieval failure:** The retriever scored irrelevant documents higher than the correct one.
3. **Context filtering failure:** The correct candidate was retrieved, but dropped before generation (e.g., context window limits).
4. **Generation failure:** The generator received the correct context, but hallucinated or ignored it.

By exposing `similarity_search_with_score` before the LLM step, you can instantly determine if a failure is a retrieval issue (search problem) or a generation issue (prompt/model problem).